# 🎤 GPT-SoVITS 全自动训练 - 达妮娅语音

**数据集**: https://huggingface.co/datasets/huanx/daniya-voice-gptsovits

按顺序运行所有单元格，全自动完成训练。

**前提**: 运行时类型 → **GPU (T4)**

## Step 0: 环境检测

In [ ]:
import os, sys

# 检测运行环境：Kaggle 或 Google Colab
if os.path.exists('/kaggle'):
    WORK_DIR = '/kaggle/working'
    print('✅ 检测到 Kaggle 环境')
else:
    WORK_DIR = '/content'
    print('✅ 检测到 Google Colab 环境')

print(f'工作目录: {WORK_DIR}')

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'当前目录: {os.getcwd()}')

## Step 1: 克隆仓库并安装依赖

In [ ]:
REPO_DIR = f'{WORK_DIR}/GPT-SoVITS'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/RVC-Boss/GPT-SoVITS.git {REPO_DIR}

os.chdir(REPO_DIR)
print(f'仓库目录: {REPO_DIR}')

!pip install -r requirements.txt 2>&1 | tail -5
!pip install huggingface_hub 2>&1 | tail -3

print('✅ Step 1 完成')

## Step 2: 下载预训练模型

In [ ]:
import zipfile
from huggingface_hub import hf_hub_download

os.chdir(REPO_DIR)

print('下载 pretrained_models.zip ...')
path = hf_hub_download(repo_id='XXXXRT/GPT-SoVITS-Pretrained',
                        filename='pretrained_models.zip',
                        local_dir='.')

print('解压中 ...')
with zipfile.ZipFile(path, 'r') as z:
    z.extractall('.')

files = os.listdir('pretrained_models')
print(f'预训练模型: {len(files)} 个文件')
for f in files:
    print(f'  {f}')

print('\n下载 G2PWModel.zip ...')
g2pw_path = hf_hub_download(repo_id='XXXXRT/GPT-SoVITS-Pretrained',
                             filename='G2PWModel.zip',
                             local_dir='.')
with zipfile.ZipFile(g2pw_path, 'r') as z:
    z.extractall('GPT_SoVITS/text/')

print('\n✅ Step 2 完成')

## Step 3: 下载数据集

In [ ]:
import shutil
from pathlib import Path
from huggingface_hub import snapshot_download

DATASET_DIR = f'{WORK_DIR}/daniya_dataset'

print('下载达妮娅数据集 ...')
snapshot_download(repo_id='huanx/daniya-voice-gptsovits',
                   repo_type='dataset',
                   local_dir=DATASET_DIR)

wav_files = list(Path(f'{DATASET_DIR}/audio').glob('*.wav'))
print(f'✅ Step 3 完成: {len(wav_files)} 个音频文件')

## Step 4: 准备数据目录

In [ ]:
import pandas as pd

os.chdir(REPO_DIR)

exp_name = 'daniya'
exp_dir = f'output/{exp_name}'

dirs = [
    f'{exp_dir}/5-wav32k',
    f'{exp_dir}/1_16k_wavs',
    f'{exp_dir}/1_32k_wavs',
    f'{exp_dir}/2a-asr',
    f'{exp_dir}/2b-feature',
    f'{exp_dir}/3-bert',
    f'{exp_dir}/4-cnhubert',
    f'{exp_dir}/6-name2semantic',
    f'{exp_dir}/logs_s1',
    f'{exp_dir}/logs_s2_v2',
    f'{exp_dir}/weights',
]
for d in dirs:
    os.makedirs(d, exist_ok=True)

audio_src = f'{DATASET_DIR}/audio'
wav_files = sorted(Path(audio_src).glob('*.wav'))
for f in wav_files:
    shutil.copy2(f, f'{exp_dir}/5-wav32k/')
    shutil.copy2(f, f'{exp_dir}/1_16k_wavs/')
    shutil.copy2(f, f'{exp_dir}/1_32k_wavs/')
print(f'复制 {len(wav_files)} 个音频')

metadata = pd.read_csv(f'{DATASET_DIR}/metadata.csv')
list_lines = []
for _, row in metadata.iterrows():
    wav_path = os.path.abspath(f'{exp_dir}/5-wav32k/{row["file"]}')
    text = str(row['text']).strip()
    list_lines.append(f'{wav_path}|daniya|zh|{text}')

inp_text_path = f'{exp_dir}/2a-asr/{exp_name}.list'
with open(inp_text_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(list_lines) + '\n')

print(f'生成 {len(list_lines)} 条训练记录')
print(f'示例: {list_lines[0][:80]}...')
print('✅ Step 4 完成')

## Step 5: 数据预处理

In [ ]:
import subprocess, json

os.chdir(REPO_DIR)

exp_dir = os.path.abspath(f'output/{exp_name}')
inp_text = os.path.abspath(f'{exp_dir}/2a-asr/{exp_name}.list')
inp_wav_dir = os.path.abspath(f'{exp_dir}/5-wav32k')

pretrained_dir = os.path.abspath('pretrained_models')
bert_dir = f'{pretrained_dir}/chinese-roberta-wwm-ext-large'
hubert_dir = f'{pretrained_dir}/chinese-hubert-base'
pretrained_s2G = f'{pretrained_dir}/gsv-v2final-pretrained/s2G2333k.pth'

# 3-get-semantic.py 需要 s2 配置文件
s2_for_preprocess = {
    'data': {'filter_length': 2048, 'hop_length': 640, 'sampling_rate': 32000,
             'n_speakers': 300, 'cleaned_text': True},
    'train': {'segment_size': 20480},
    'model': {'version': 'v2'},
}
s2_preprocess_path = f'{WORK_DIR}/s2_preprocess.json'
with open(s2_preprocess_path, 'w') as f:
    json.dump(s2_for_preprocess, f)

base_env = {
    **os.environ,
    'inp_text': inp_text,
    'inp_wav_dir': inp_wav_dir,
    'exp_name': exp_name,
    'opt_dir': exp_dir,
    'i_part': '0',
    'all_parts': '1',
    'is_half': 'True',
}

# 5a: 提取文本特征
print('=== 5a: 提取文本特征 ===')
env1 = {**base_env, 'bert_pretrained_dir': bert_dir}
r = subprocess.run([sys.executable, '-s', 'GPT_SoVITS/prepare_datasets/1-get-text.py'],
                    env=env1, capture_output=True, text=True)
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
else:
    print('✅ 1-get-text 完成')

# 5b: 提取 HuBERT 特征
print('\n=== 5b: 提取音频特征 ===')
env2 = {**base_env, 'cnhubert_base_dir': hubert_dir}
r = subprocess.run([sys.executable, '-s', 'GPT_SoVITS/prepare_datasets/2-get-hubert-wav32k.py'],
                    env=env2, capture_output=True, text=True)
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
else:
    print('✅ 2-get-hubert 完成')

# 5c: 提取语义
print('\n=== 5c: 提取语义 ===')
env3 = {
    **base_env,
    'cnhubert_base_dir': hubert_dir,
    'pretrained_s2G': pretrained_s2G,
    's2config_path': os.path.abspath(s2_preprocess_path),
}
r = subprocess.run([sys.executable, '-s', 'GPT_SoVITS/prepare_datasets/3-get-semantic.py'],
                    env=env3, capture_output=True, text=True)
if r.returncode != 0:
    print('STDERR:', r.stderr[-1000:])
else:
    print('✅ 3-get-semantic 完成')

# 重命名语义文件（3-get-semantic 输出 6-name2semantic-0.tsv）
src_semantic = f'{exp_dir}/6-name2semantic-0.tsv'
dst_semantic = f'{exp_dir}/6-name2semantic.tsv'
if os.path.exists(src_semantic):
    shutil.copy2(src_semantic, dst_semantic)
    print(f'复制语义文件: {src_semantic} → {dst_semantic}')

# 生成 2-name2text.txt（GPT 训练需要）
s1_dir = f'{exp_dir}/logs_s1'
os.makedirs(s1_dir, exist_ok=True)
name2text_path = f'{s1_dir}/2-name2text.txt'
with open(inp_text, 'r', encoding='utf-8') as fin, \
     open(name2text_path, 'w', encoding='utf-8') as fout:
    for line in fin:
        parts = line.strip().split('|')
        if len(parts) >= 4:
            wav_name = os.path.basename(parts[0]).replace('.wav', '')
            text = parts[3]
            fout.write(f'{wav_name}\t{text}\n')
print(f'生成 {name2text_path}')

# 检查产出
print('\n=== 产出检查 ===')
for subdir in ['2a-asr', '3-bert', '4-cnhubert', '5-wav32k', '6-name2semantic']:
    path = f'{exp_dir}/{subdir}'
    if os.path.exists(path):
        print(f'  {subdir}: {len(os.listdir(path))} 个文件')
    else:
        print(f'  {subdir}: ⚠️ 目录不存在')

print('\n✅ Step 5 完成')

## Step 6: 训练 SoVITS

In [ ]:
os.chdir(REPO_DIR)

exp_dir = os.path.abspath(f'output/{exp_name}')

with open('GPT_SoVITS/configs/s2.json') as f:
    s2 = json.load(f)

s2['train']['batch_size'] = 4
s2['train']['epochs'] = 200
s2['train']['save_every_epoch'] = 50
s2['train']['if_save_latest'] = True
s2['train']['if_save_every_weights'] = True
s2['train']['fp16_run'] = True
s2['train']['grad_ckpt'] = True
s2['train']['gpu_numbers'] = '0'
s2['train']['pretrained_s2G'] = os.path.abspath('pretrained_models/gsv-v2final-pretrained/s2G2333k.pth')
s2['train']['pretrained_s2D'] = os.path.abspath('pretrained_models/gsv-v2final-pretrained/s2D2333k.pth')
s2['data']['exp_dir'] = exp_dir
s2['s2_ckpt_dir'] = exp_dir
s2['name'] = exp_name
s2['model']['version'] = 'v2'

tmp = f'{WORK_DIR}/s2_config.json'
with open(tmp, 'w') as f:
    json.dump(s2, f, indent=2)

print('SoVITS 训练: batch_size=4, epochs=200, save_every_epoch=50')
!python -s GPT_SoVITS/s2_train.py --config {tmp}

print('✅ Step 6 完成')

## Step 7: 训练 GPT

In [ ]:
import yaml

os.chdir(REPO_DIR)

exp_dir = os.path.abspath(f'output/{exp_name}')
s1_dir = f'{exp_dir}/logs_s1'

s1 = {
    'train_semantic_path': f'{s1_dir}/6-name2semantic.tsv',
    'train_phoneme_path': f'{s1_dir}/2-name2text.txt',
    'output_dir': s1_dir,
    'train': {
        'seed': 1234,
        'epochs': 100,
        'batch_size': 8,
        'save_every_n_epoch': 1,
        'if_save_latest': True,
        'if_save_every_weights': True,
        'half_weights_save_dir': f'{exp_dir}/weights',
        'exp_name': exp_name,
        'precision': '16-mixed',
        'gradient_clip': 1.0,
    },
    'optimizer': {
        'lr': 0.01,
        'lr_init': 0.00001,
        'lr_end': 0.0001,
        'warmup_steps': 2000,
        'decay_steps': 40000,
    },
    'data': {
        'max_eval_sample': 8,
        'max_sec': 54,
        'num_workers': 4,
        'pad_val': 1024,
    },
    'model': {
        'vocab_size': 1025,
        'phoneme_vocab_size': 512,
        'embedding_dim': 512,
        'hidden_dim': 512,
        'head': 16,
        'linear_units': 2048,
        'n_layer': 24,
        'dropout': 0,
        'EOS': 1024,
        'random_bert': 0,
    },
    'inference': {
        'top_k': 5,
    },
}

tmp = f'{WORK_DIR}/s1_config.yaml'
with open(tmp, 'w') as f:
    yaml.dump(s1, f, default_flow_style=False)

print('GPT 训练: batch_size=8, epochs=100, save_every_n_epoch=1')
!python -s GPT_SoVITS/s1_train.py --config_file {tmp}

print('✅ Step 7 完成')

## Step 8: 打包下载

In [ ]:
os.chdir(REPO_DIR)

print('=== 训练产出 ===')
for root, dirs, files in os.walk('output'):
    for f in sorted(files):
        if f.endswith(('.pth', '.ckpt')):
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / 1024 / 1024
            print(f'  {fpath} ({size_mb:.1f} MB)')

MODEL_FILE = f'{WORK_DIR}/daniya_gptsovits_model.tar.gz'

!tar -czf {MODEL_FILE} output/
!ls -lh {MODEL_FILE}

from IPython.display import FileLink, display
display(FileLink(MODEL_FILE))
print('\n✅ 完成! 点击链接下载模型')